In [ ]:
#Nouveau brut force

In [2]:
import pandas as pd
import numpy as np
import unicodedata
from itertools import combinations
 
# =========================
# 1) Chargement
# =========================
df = pd.read_excel("/Users/thibaultdautheville/Downloads/REI département total.xlsx")


In [31]:

# =========================
# 2) Normalisation des noms de départements
# =========================
def normalize(s):
    s = str(s).strip().upper()
    s = "".join(
        c for c in unicodedata.normalize("NFD", s)
        if unicodedata.category(c) != "Mn"
    )
    s = s.replace("-", " ").replace("'", " ")
    s = " ".join(s.split())
    return s
 
df["DEP_NORM"] = df["LIBELLÉ DÉPARTEMENT"].apply(normalize)
 

In [32]:

# =========================
# 3) Cibles connues (part entreprise, en k€)
# =========================
targets = {
    "AIN": 88307,
    "ARDECHE": 48610,
    "SAVOIE": 125779,
    "HAUTE SAVOIE": 106718,
    "PARIS": 750648,
    "CALVADOS": 107390,
    "CREUSE": 13378,
    "CHARENTE MARITIME": 73052,
    "PYRENEES ATLANTIQUES": 80070,
    "ALPES MARITIMES": 217386,
    "MARTINIQUE": 52999,
    "LA REUNION": 71438,
    "VAUCLUSE": 89228,
    "TARN ET GARONNE": 36112,
    "VIENNE": 48609,
    "EURE": 80602,
    "YVELINES": 251590,
    "VOSGES": 55057,
    "HAUTE CORSE": 18164,
    "MORBIHAN": 85428,
    "YONNE": 51620,
}


In [ ]:
 
# =========================
# 4) Variables FB à tester
# =========================
base_cols = [
    "FB - COMMUNE / MONTANT RÉEL",
    "FB - GFP / MONTANT RÉEL",
    "FB - SYNDICATS ET ORG. ASSIMILÉS / MONTANT RÉEL",  # SYNDICATS avec S
]
 
candidate_cols = [
    "FB - TSE / MONTANT RÉEL",
    "FB - TSE AUTRES / MONTANT RÉEL",
    "FB - GEMAPI / MONTANT RÉEL INTERCOMMUNALITÉ",
    "FB - TASA / MONTANT RÉEL",
    "FB - SOMME DES ALLOCATIONS COMPENSATRICES / COMMUNE",
    "FB - SOMME DES ALLOCATIONS COMPENSATRICES / EPCI",
    "FB - SOMME DES ALLOCATIONS COMPENSATRICES / REG",
    "FB - ALLOCATION COMPENSATRICE - EXONÉRATIONS DE LONGUE DUREE / COMMUNE",
    "FB - ALLOCATION COMPENSATRICE - EXONÉRATIONS DE LONGUE DUREE / INTERCOMMUNALITÉ",
    "FB - TEOM / TAUX PLEIN - MONTANT NET LISSE",
    "FB - TEOM / TAUX REDUIT A - MONTANT NET",
    "FB - TEOM / TAUX REDUIT B - MONTANT NET",
    "FB - TEOM / TAUX REDUIT C -  MONTANT NET",
    "FB - TEOM / TAUX REDUIT D - MONTANT NET",
    "FB - TEOM / TAUX REDUIT E - MONTANT NET",
    "FB - TEOM INCITATIVE / MONTANT RÉEL / COMMUNE",
    "FB - TEOM INCITATIVE / MONTANT RÉEL / SYNDICAT",
    "FB - TEOM INCITATIVE / MONTANT RÉEL / GFP",
    "FB - TEOM / MONTANT RÉEL TOTAL",
]
 
all_cols = base_cols + candidate_cols
 
# sécurisation minimale
for c in all_cols:
    if c in df.columns:
        df[c] = pd.to_numeric(df[c], errors="coerce").fillna(0)
 

In [34]:

# =========================
# 5) Agrégation départementale
# =========================
dept = (
    df.groupby(["DEP_NORM", "LIBELLÉ DÉPARTEMENT"], as_index=False)[all_cols]
      .sum()
)
 
dept = dept[dept["DEP_NORM"].isin(targets.keys())].copy()
dept["CIBLE_k"] = dept["DEP_NORM"].map(targets)
 

In [ ]:

# =========================
# 6) Brute force amélioré
# =========================
max_added = 6   # augmente à 6 si tu veux, mais ça rallonge
rows = []
 
for r in range(0, max_added + 1):
    for combo in combinations(candidate_cols, r):
        cols_used = base_cols + list(combo)
 
        pred_k = dept[cols_used].sum(axis=1) * 0.22 / 1000
        err_k = pred_k - dept["CIBLE_k"]
        abs_err_k = err_k.abs()
        abs_pct_err = abs_err_k / dept["CIBLE_k"] * 100
 
        rows.append({
            "nb_blocs_ajoutes": r,
            "combinaison": " + ".join(combo) if combo else "[BASE]",
            "nb_departements": len(dept),
            "MAE_k": round(abs_err_k.mean(), 1),
            "MAPE_%": round(abs_pct_err.mean(), 2),
            "MEDAPE_%": round(abs_pct_err.median(), 2),
            "MAX_ERR_%": round(abs_pct_err.max(), 2),
            "RMSE_k": round(np.sqrt((err_k ** 2).mean()), 1),
        })
 
scores = pd.DataFrame(rows).sort_values(
    ["MAPE_%", "MAX_ERR_%", "MAE_k", "nb_blocs_ajoutes"]
).reset_index(drop=True)
 
print("=== TOP 20 des meilleures combinaisons ===")
print(scores.head(20))
 

=== TOP 20 des meilleures combinaisons ===
    nb_blocs_ajoutes                                        combinaison  \
0                  5  FB - TSE AUTRES / MONTANT RÉEL + FB - GEMAPI /...   
1                  5  FB - TSE AUTRES / MONTANT RÉEL + FB - GEMAPI /...   
2                  4  FB - TSE AUTRES / MONTANT RÉEL + FB - GEMAPI /...   
3                  5  FB - TSE AUTRES / MONTANT RÉEL + FB - GEMAPI /...   
4                  5  FB - TSE AUTRES / MONTANT RÉEL + FB - GEMAPI /...   
5                  5  FB - TSE AUTRES / MONTANT RÉEL + FB - GEMAPI /...   
6                  5  FB - TSE AUTRES / MONTANT RÉEL + FB - GEMAPI /...   
7                  5  FB - TSE AUTRES / MONTANT RÉEL + FB - GEMAPI /...   
8                  5  FB - TSE AUTRES / MONTANT RÉEL + FB - GEMAPI /...   
9                  5  FB - GEMAPI / MONTANT RÉEL INTERCOMMUNALITÉ + ...   
10                 4  FB - GEMAPI / MONTANT RÉEL INTERCOMMUNALITÉ + ...   
11                 5  FB - GEMAPI / MONTANT RÉEL INTERCOM

In [35]:

# =========================
# 7) Détail du meilleur modèle résultats partagés
# =========================
best_combo = scores.loc[0, "combinaison"]
if best_combo == "[BASE]":
    best_cols = base_cols
else:
    best_cols = base_cols + best_combo.split(" + ")
 
detail = dept[["DEP_NORM", "LIBELLÉ DÉPARTEMENT", "CIBLE_k"]].copy()
detail["PRED_k"] = dept[best_cols].sum(axis=1) * 0.22 / 1000
detail["ECART_k"] = detail["PRED_k"] - detail["CIBLE_k"]
detail["ECART_%"] = (detail["ECART_k"].abs() / detail["CIBLE_k"] * 100).round(2)
detail = detail.sort_values("ECART_%")
 
print("\n=== Meilleure combinaison trouvée ===")
print(best_combo)
print("\n=== Détail par département (meilleur modèle) ===")
print(detail[["LIBELLÉ DÉPARTEMENT", "CIBLE_k", "PRED_k", "ECART_k", "ECART_%"]])


=== Meilleure combinaison trouvée ===
FB - TSE AUTRES / MONTANT RÉEL + FB - GEMAPI / MONTANT RÉEL INTERCOMMUNALITÉ + FB - TASA / MONTANT RÉEL + FB - SOMME DES ALLOCATIONS COMPENSATRICES / COMMUNE + FB - ALLOCATION COMPENSATRICE - EXONÉRATIONS DE LONGUE DUREE / INTERCOMMUNALITÉ

=== Détail par département (meilleur modèle) ===
      LIBELLÉ DÉPARTEMENT  CIBLE_k  PRED_k  ECART_k  ECART_%
22                 CREUSE    13378   13354      -24        0
13               CALVADOS   107390  104876    -2514        2
65             MARTINIQUE    52999   51656    -1343        3
42           HAUTE-SAVOIE   106718  110636     3918        4
37            HAUTE-CORSE    18164   17082    -1082        6
99                 VIENNE    48609   51870     3261        7
28                   EURE    80602   75094    -5508        7
102              YVELINES   251590  230988   -20602        8
97               VAUCLUSE    89228   80900    -8328        9
79   PYRENEES-ATLANTIQUES    80070   88653     8583       11


In [43]:
# =========================
# 6) Brute force amélioré
# avec +/- colonnes
# =========================
from itertools import combinations, product

max_added = 4  # réduis si trop lent (2^n combinaisons de signes)
rows = []

for r in range(0, max_added + 1):
    for combo in combinations(candidate_cols, r):
        # Pour chaque combinaison, on teste tous les signes +/-
        for signs in product([1, -1], repeat=len(combo)):
            
            # Construire la prédiction
            pred_k = dept[base_cols].sum(axis=1).copy()
            
            for col, sign in zip(combo, signs):
                pred_k += sign * dept[col]
            
            pred_k = pred_k * 0.22 / 1000
            
            err_k = pred_k - dept["CIBLE_k"]
            abs_err_k = err_k.abs()
            abs_pct_err = abs_err_k / dept["CIBLE_k"] * 100

            # Libellé lisible : "+col" ou "-col"
            combo_label = " ".join(
                f"+{col}" if s == 1 else f"-{col}"
                for col, s in zip(combo, signs)
            )

            rows.append({
                "nb_blocs_ajoutes": r,
                "combinaison": combo_label if combo_label else "[BASE]",
                "MAE_k": round(abs_err_k.mean(), 1),
                "MAPE_%": round(abs_pct_err.mean(), 2),
                "MEDAPE_%": round(abs_pct_err.median(), 2),
                "MAX_ERR_%": round(abs_pct_err.max(), 2),
                "RMSE_k": round(np.sqrt((err_k ** 2).mean()), 1),
            })

scores = pd.DataFrame(rows).sort_values(
    ["MAPE_%", "MAX_ERR_%", "MAE_k", "nb_blocs_ajoutes"]
).reset_index(drop=True)

print("=== TOP 20 des meilleures combinaisons ===")
print(scores.head(20))


=== TOP 20 des meilleures combinaisons ===
    nb_blocs_ajoutes                                        combinaison  \
0                  4  +FB - GEMAPI / MONTANT RÉEL INTERCOMMUNALITÉ +...   
1                  4  +FB - TASA / MONTANT RÉEL +FB - SOMME DES ALLO...   
2                  4  +FB - TSE AUTRES / MONTANT RÉEL +FB - SOMME DE...   
3                  4  +FB - GEMAPI / MONTANT RÉEL INTERCOMMUNALITÉ +...   
4                  4  +FB - SOMME DES ALLOCATIONS COMPENSATRICES / C...   
5                  4  +FB - TSE AUTRES / MONTANT RÉEL +FB - GEMAPI /...   
6                  4  +FB - GEMAPI / MONTANT RÉEL INTERCOMMUNALITÉ +...   
7                  4  +FB - SOMME DES ALLOCATIONS COMPENSATRICES / C...   
8                  4  +FB - SOMME DES ALLOCATIONS COMPENSATRICES / C...   
9                  3  +FB - SOMME DES ALLOCATIONS COMPENSATRICES / C...   
10                 4  +FB - SOMME DES ALLOCATIONS COMPENSATRICES / C...   
11                 4  +FB - SOMME DES ALLOCATIONS COMPENS

In [39]:
# Renommage temporaire : " - " -> "__"
rename_map = {c: c.replace(" - ", "__") for c in dept.columns}
rename_back = {v: k for k, v in rename_map.items()}

dept_r = dept.rename(columns=rename_map)

base_cols_r = [c.replace(" - ", "__") for c in base_cols]
candidate_cols_r = [c.replace(" - ", "__") for c in candidate_cols]


In [42]:
# =========================
# 7) Détail du meilleur modèle
# =========================
best_combo = scores.loc[0, "combinaison"]

pred_k = dept_r[base_cols_r].sum(axis=1).copy()

if best_combo != "[BASE]":
    tokens = re.findall(r'([+-])([^+-]+)', best_combo)
    for sign_str, col in tokens:
        col = col.strip()
        sign = 1 if sign_str == "+" else -1
        if col in dept_r.columns:
            pred_k += sign * dept_r[col]
        else:
            print(f"⚠️ Colonne non trouvée : '{col}'")

pred_k = pred_k * 0.22 / 1000

detail = dept[["DEP_NORM", "LIBELLÉ DÉPARTEMENT", "CIBLE_k"]].copy()
detail["PRED_k"] = pred_k.values
detail["ECART_k"] = detail["PRED_k"] - detail["CIBLE_k"]
detail["ECART_%"] = (detail["ECART_k"].abs() / detail["CIBLE_k"] * 100).round(2)
detail = detail.sort_values("ECART_%")

# Affichage avec nom de colonne lisible
best_combo_display = best_combo.replace("__", " - ")
print("\n=== Meilleure combinaison trouvée ===")
print(best_combo_display)
print(f"\nMAPE : {scores.loc[0, 'mape']:.2f}%")
print("\n=== Détail par département ===")
print(detail[["LIBELLÉ DÉPARTEMENT", "CIBLE_k", "PRED_k", "ECART_k", "ECART_%"]].to_string())


⚠️ Colonne non trouvée : 'FB'
⚠️ Colonne non trouvée : 'GEMAPI / MONTANT RÉEL INTERCOMMUNALITÉ'
⚠️ Colonne non trouvée : 'FB'
⚠️ Colonne non trouvée : 'SOMME DES ALLOCATIONS COMPENSATRICES / COMMUNE'
⚠️ Colonne non trouvée : 'FB'
⚠️ Colonne non trouvée : 'TEOM / TAUX REDUIT A'
⚠️ Colonne non trouvée : 'MONTANT NET'
⚠️ Colonne non trouvée : 'FB'
⚠️ Colonne non trouvée : 'TEOM INCITATIVE / MONTANT RÉEL / GFP'

=== Meilleure combinaison trouvée ===
+FB - GEMAPI / MONTANT RÉEL INTERCOMMUNALITÉ +FB - SOMME DES ALLOCATIONS COMPENSATRICES / COMMUNE -FB - TEOM / TAUX REDUIT A - MONTANT NET +FB - TEOM INCITATIVE / MONTANT RÉEL / GFP


KeyError: 'mape'

In [40]:
# =========================
# 7) Détail du meilleur modèle (avec +/- colonnes)
# =========================
best_combo = scores.loc[0, "combinaison"]

pred_k = dept[base_cols].sum(axis=1).copy()

if best_combo != "[BASE]":
    import re
    # Trouver tous les tokens : signe + nom de colonne
    tokens = re.findall(r'([+-])([^+-]+)', best_combo)
    for sign_str, col in tokens:
        col = col.strip()
        sign = 1 if sign_str == "+" else -1
        if col in dept.columns:
            pred_k += sign * dept[col]
        else:
            print(f"⚠️ Colonne non trouvée : '{col}'")

pred_k = pred_k * 0.22 / 1000

detail = dept[["DEP_NORM", "LIBELLÉ DÉPARTEMENT", "CIBLE_k"]].copy()
detail["PRED_k"] = pred_k
detail["ECART_k"] = detail["PRED_k"] - detail["CIBLE_k"]
detail["ECART_%"] = (detail["ECART_k"].abs() / detail["CIBLE_k"] * 100).round(2)
detail = detail.sort_values("ECART_%")

print("\n=== Meilleure combinaison trouvée ===")
print(best_combo)
print("\n=== Détail par département (meilleur modèle) ===")
print(detail[["LIBELLÉ DÉPARTEMENT", "CIBLE_k", "PRED_k", "ECART_k", "ECART_%"]])


⚠️ Colonne non trouvée : 'FB'
⚠️ Colonne non trouvée : 'GEMAPI / MONTANT RÉEL INTERCOMMUNALITÉ'
⚠️ Colonne non trouvée : 'FB'
⚠️ Colonne non trouvée : 'SOMME DES ALLOCATIONS COMPENSATRICES / COMMUNE'
⚠️ Colonne non trouvée : 'FB'
⚠️ Colonne non trouvée : 'TEOM / TAUX REDUIT A'
⚠️ Colonne non trouvée : 'MONTANT NET'
⚠️ Colonne non trouvée : 'FB'
⚠️ Colonne non trouvée : 'TEOM INCITATIVE / MONTANT RÉEL / GFP'

=== Meilleure combinaison trouvée ===
+FB - GEMAPI / MONTANT RÉEL INTERCOMMUNALITÉ +FB - SOMME DES ALLOCATIONS COMPENSATRICES / COMMUNE -FB - TEOM / TAUX REDUIT A - MONTANT NET +FB - TEOM INCITATIVE / MONTANT RÉEL / GFP

=== Détail par département (meilleur modèle) ===
      LIBELLÉ DÉPARTEMENT  CIBLE_k  PRED_k  ECART_k  ECART_%
99                 VIENNE    48609   48161     -448        1
42           HAUTE-SAVOIE   106718  104992    -1726        2
92        TARN-ET-GARONNE    36112   36969      857        2
79   PYRENEES-ATLANTIQUES    80070   83198     3128        4
22           

In [44]:
print(df.columns.tolist())



['DÉPARTEMENT', 'DIRECTION', 'CODE COMMUNE', 'COMMUNE RECENSÉE (R si recensée)', 'LIBELLÉ COMMUNE', 'Numéro national du groupement', "Numéro SIREN de l'EPCI", 'Libellé du Groupement', "Option fiscale de l'EPCI (FPA, FPZ ou FPU)", 'Forme juridique EPCI (CC, CA, CU ou Mét)', 'LIBELLÉ DÉPARTEMENT', 'LIBELLÉ RÉGION', "FNB - FRAIS D'ASSIETTE, DEGREVEMENT, NON VALEURS", 'FNB - COMMUNE / BASE NETTE', 'FNB - COMMUNE / TAUX NET', 'FNB - COMMUNE / TAUX VOTÉ', 'FNB - COMMUNE / MONTANT RÉEL', "FNB - COMMUNE / NOMBRE D'ARTICLES", 'FNB - SYNDICATS ET ORG. ASSIMILÉS / BASE NETTE', 'FNB - SYNDICATS ET ORG. ASSIMILÉS / TAUX NET', 'FNB - SYNDICATS ET ORG. ASSIMILÉS / MONTANT RÉEL', 'FNB - GFP / BASE NETTE', 'FNB - GFP / TAUX NET', 'FNB - GFP / TAUX VOTÉ', 'FNB - GFP / MONTANT RÉEL', 'TAFNB - BASE TAXABLE COMMUNALE', 'TAFNB - COMMUNE / TAUX NET', 'TAFNB - COMMUNE / MONTANT RÉEL', 'TAFNB - BASE TAXABLE GFP', 'TAFNB - GFP / TAUX NET', 'TAFNB - GFP / MONTANT RÉEL', 'TAFNB - BASE TAXABLE MÉTROPOLE DU GRAND P